# MS-Dialog Data Preparation

Extracts 100 high-quality tech-support dialogs from **MSDialog-Intent** for the clarifying-question uncertainty experiment.

## Experiment mapping (parallel to MedQA)

| MedQA field | MS-Dialog equivalent |
|---|---|
| `ehr_summary` | `original_question` (user's OQ utterance) |
| `question` | fixed prompt: *"What is the most helpful solution to this user's problem?"* |
| `options` | N/A — open-ended, not MCQ |
| `simulator_context` | User FD utterances + Agent CQ utterances (hidden context) |
| `correct_answer` | `accepted_answer` (utterance(s) with `is_answer=1`) |

## Tag legend
| Tag | Meaning |
|---|---|
| **OQ** | Original Question — the user's problem statement |
| **PA** | Potential Answer — agent's proposed solution |
| **FD** | Further Details — user elaborates on the problem |
| **CQ** | Clarifying Question — agent (or user) asks for more info |
| **IR** | Information Request — similar to CQ |
| **GG** | Gratitude / Greeting |
| **PF** | Positive Feedback |
| **NF** | Negative Feedback |

## Output
`datasets/ms-dialog/msdialog_100.jsonl` — one JSON object per line with fields:
```
case_id, title, category, original_question,
simulator_context, accepted_answer, n_utterances, has_fd, has_agent_cq
```

In [1]:
import sys
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT       = Path("../../").resolve()
DATA_DIR   = ROOT / "datasets" / "ms-dialog"
OUT_DIR    = DATA_DIR

INTENT_JSON = DATA_DIR / "MSDialog-Intent.json"
OUTPUT_JSONL = OUT_DIR / "msdialog_100.jsonl"

N_SAMPLES   = 100   # total examples to extract
RANDOM_SEED = 42

sys.path.insert(0, str(ROOT))
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT        : {ROOT}")
print(f"Input JSON  : {INTENT_JSON}")
print(f"Output JSONL: {OUTPUT_JSONL}")

ROOT        : D:\final_project\pilot_study
Input JSON  : D:\final_project\pilot_study\datasets\ms-dialog\MSDialog-Intent.json
Output JSONL: D:\final_project\pilot_study\datasets\ms-dialog\msdialog_100.jsonl


## 1. Load Dataset

In [2]:
import json
from collections import Counter, defaultdict

with open(INTENT_JSON, encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Total dialogs loaded: {len(raw_data)}")

# ── Inspect tag distribution ──────────────────────────────────────────────────
tag_counts = Counter()
for dialog in raw_data.values():
    for u in dialog.get("utterances", []):
        tags = u.get("tags", "").split() if isinstance(u.get("tags", ""), str) else u.get("tags", [])
        tag_counts.update(tags)

print("\nTag distribution across all utterances:")
for tag, count in tag_counts.most_common():
    print(f"  {tag:5s}: {count:5d}")

Total dialogs loaded: 2199

Tag distribution across all utterances:
  GG   :  4004
  PA   :  3980
  FD   :  2481
  OQ   :  2339
  PF   :  1076
  IR   :  1076
  FQ   :   877
  NF   :   756
  CQ   :   749
  RQ   :   606
  JK   :   256
  O    :   149


## 2. Define Extraction Helpers

**`get_tags(u)`** — normalises both string and list tag formats.

**`extract_fields(dialog_id, dialog)`** — returns a structured dict or `None` if the dialog fails quality filters.

### Quality filters
1. Must have at least one **OQ** utterance with ≥ 15 words  
2. Must have at least one **accepted answer** (`is_answer == 1`)  
3. Must have ≥ 3 utterances (enough context for a simulator to respond to a follow-up CQ)

### Simulator context construction
The simulator context gives the model-as-simulator enough information to answer any clarifying question the clinician/model generates. It is built from:
- All **User FD** utterances — the user's own further details  
- All **Agent CQ** utterances — questions the support agent originally asked  

These two sources together let the simulator roleplay as the user with full knowledge of the problem details and what information was sought. They are separated by `---` so the simulator knows the source.

### Accepted-answer fallback chain
| Priority | Rule |
|---|---|
| 1 | First PA utterance with `is_answer == 1` (explicit mark) |
| 2 | All PA utterances with `is_answer == 1` joined if multiple |
| 3 | Last PA utterance from Agent (no explicit mark — use heuristic) |

In [3]:
def get_tags(utterance: dict) -> list[str]:
    """Return list of tags, handling both str and list formats."""
    raw = utterance.get("tags", "")
    if isinstance(raw, list):
        return raw
    return raw.split() if raw else []


def extract_fields(dialog_id: str, dialog: dict) -> dict | None:
    """Parse one dialog into the experiment record schema.
    
    Returns None if the dialog does not meet quality thresholds.

    Simulator context construction
    --------------------------------
    Goal: give the simulator enough hidden information to answer any CQ
    our model generates about the user's problem — without leaking the solution.

    Included:
      - User FD utterances  : further details the user volunteered
      - Agent CQ utterances : ONLY pure CQ — must NOT carry a PA tag

    Excluded from simulator context:
      - Any Agent utterance also tagged PA (hybrid CQ+PA) — these ARE the
        accepted answer in 24 % of dialogs and would constitute direct leakage
      - Accepted-answer utterances (identified before building context)
    """
    utts = dialog.get("utterances", [])

    if len(utts) < 3:
        return None

    # ── Pass 1: identify accepted-answer utterances ───────────────────────────
    # We need to know which utterances are answers BEFORE building context,
    # so we can exclude them even if they carry a CQ tag.
    answer_texts    = []   # PA utterances with is_answer == 1
    pa_fallback     = []   # All Agent PA utterances (for fallback)

    for u in utts:
        tags  = get_tags(u)
        text  = u.get("utterance", "").strip()
        actor = u.get("actor_type", "")
        marked = u.get("is_answer", 0) == 1

        if not text:
            continue

        if "PA" in tags and actor == "Agent":
            pa_fallback.append(text)
            if marked:
                answer_texts.append(text)

    # ── Resolve accepted answer ────────────────────────────────────────────────
    if answer_texts:
        accepted_answer = "\n\n".join(answer_texts)
        answer_source   = "explicit_is_answer"
    elif pa_fallback:
        accepted_answer = pa_fallback[-1]
        answer_source   = "fallback_last_pa"
    else:
        return None  # no usable answer

    answer_set = set(answer_texts if answer_texts else [pa_fallback[-1]])

    # ── Pass 2: build OQ + simulator context (answer utterances excluded) ─────
    oq_texts       = []
    user_fd_texts  = []
    agent_cq_texts = []

    for u in utts:
        tags  = get_tags(u)
        text  = u.get("utterance", "").strip()
        actor = u.get("actor_type", "")

        if not text:
            continue

        # ── OQ ────────────────────────────────────────────────────────────────
        if "OQ" in tags and actor == "User":
            oq_texts.append(text)

        # ── User FD (safe: users never produce accepted answers) ──────────────
        if "FD" in tags and actor == "User":
            user_fd_texts.append(text)

        # ── Agent CQ — STRICT: exclude any utterance also tagged PA ──────────
        # Rationale: CQ+PA hybrid utterances are the accepted answer in 24/100
        # cases. Including them in simulator context would directly leak the
        # solution to the simulator and, transitively, to the model.
        if "CQ" in tags and actor == "Agent" and "PA" not in tags:
            # Belt-and-suspenders: also skip if text is already in answer_set
            if text not in answer_set:
                agent_cq_texts.append(text)

    # ── Quality gates ──────────────────────────────────────────────────────────
    if not oq_texts:
        return None

    original_question = " ".join(oq_texts)
    if len(original_question.split()) < 15:
        return None

    # ── Simulator context ──────────────────────────────────────────────────────
    sim_parts = []
    if user_fd_texts:
        sim_parts.append(
            "[Further details provided by the user]\n"
            + "\n\n".join(user_fd_texts)
        )
    if agent_cq_texts:
        sim_parts.append(
            "[Clarifying questions the support agent previously asked]\n"
            + "\n\n".join(agent_cq_texts)
        )
    simulator_context = "\n\n---\n\n".join(sim_parts) if sim_parts else ""

    return {
        "case_id":            dialog_id,
        "title":              dialog.get("title", ""),
        "category":           dialog.get("category", ""),
        "original_question":  original_question,
        "simulator_context":  simulator_context,
        "accepted_answer":    accepted_answer,
        "answer_source":      answer_source,
        "n_utterances":       len(utts),
        "has_fd":             bool(user_fd_texts),
        "has_agent_cq":       bool(agent_cq_texts),
    }


print("Helper functions defined.")
print("Fix applied: Agent CQ utterances with PA tag are excluded from simulator context.")


Helper functions defined.
Fix applied: Agent CQ utterances with PA tag are excluded from simulator context.


## 3. Filter Quality Dialogs

In [4]:
all_records = []
skip_reasons = Counter()

for dialog_id, dialog in raw_data.items():
    rec = extract_fields(dialog_id, dialog)
    if rec is None:
        skip_reasons["failed_quality_filter"] += 1
    else:
        all_records.append(rec)

print(f"Dialogs passing quality filter : {len(all_records)} / {len(raw_data)}")
print(f"Skipped                        : {skip_reasons['failed_quality_filter']}")

# ── Context richness breakdown ────────────────────────────────────────────────
has_fd_count      = sum(r["has_fd"]        for r in all_records)
has_cq_count      = sum(r["has_agent_cq"]  for r in all_records)
has_both          = sum(r["has_fd"] and r["has_agent_cq"] for r in all_records)
explicit_answers  = sum(r["answer_source"] == "explicit_is_answer" for r in all_records)
fallback_answers  = sum(r["answer_source"] == "fallback_last_pa"   for r in all_records)

print(f"\nSimulator context richness:")
print(f"  Has user FD                  : {has_fd_count} ({has_fd_count/len(all_records):.1%})")
print(f"  Has agent CQ                 : {has_cq_count} ({has_cq_count/len(all_records):.1%})")
print(f"  Has both FD + CQ             : {has_both} ({has_both/len(all_records):.1%})")
print(f"\nAnswer source:")
print(f"  Explicit is_answer==1        : {explicit_answers} ({explicit_answers/len(all_records):.1%})")
print(f"  Fallback (last PA)           : {fallback_answers} ({fallback_answers/len(all_records):.1%})")

# ── Category distribution ─────────────────────────────────────────────────────
cat_counts = Counter(r["category"] for r in all_records)
print(f"\nTop 15 categories:")
for cat, cnt in cat_counts.most_common(15):
    print(f"  {cat:35s}: {cnt}")

Dialogs passing quality filter : 1995 / 2199
Skipped                        : 204

Simulator context richness:
  Has user FD                  : 920 (46.1%)
  Has agent CQ                 : 190 (9.5%)
  Has both FD + CQ             : 92 (4.6%)

Answer source:
  Explicit is_answer==1        : 1809 (90.7%)
  Fallback (last PA)           : 186 (9.3%)

Top 15 categories:
  PowerPoint                         : 190
  Excel                              : 177
  Bing_Apps                          : 152
  Bing_Search                        : 151
  Windows_7                          : 138
  Word                               : 131
  Bing_Maps                          : 131
  Skype_iOS                          : 123
  Skype_Android                      : 113
  Skype_Mac                          : 92
  Windows_RT_8.1                     : 92
  Windows_8.1                        : 91
  Skype_Windows_Desktop              : 86
  Apps_Windows_10                    : 79
  Skype_Windows_10                

## 4. Stratified Sampling — 100 Examples

We prioritise dialogs with **both FD and agent CQ** (richest simulator context) then fill remaining slots proportionally from categories to preserve domain variety.

Sampling priority:
1. Dialogs with FD + CQ (richest context) — proportional by category  
2. Dialogs with at least FD or CQ — proportional by category  
3. Remaining (OQ + answer only) — proportional by category

In [5]:
import random
random.seed(RANDOM_SEED)

# ── Tier assignment ───────────────────────────────────────────────────────────
tier1 = [r for r in all_records if r["has_fd"] and r["has_agent_cq"]]  # richest
tier2 = [r for r in all_records if (r["has_fd"] or r["has_agent_cq"]) and not (r["has_fd"] and r["has_agent_cq"])]
tier3 = [r for r in all_records if not r["has_fd"] and not r["has_agent_cq"]]

print(f"Tier 1 (FD + CQ)  : {len(tier1)}")
print(f"Tier 2 (FD or CQ) : {len(tier2)}")
print(f"Tier 3 (OQ only)  : {len(tier3)}")


def stratified_sample(pool: list, n: int, category_key: str = "category", seed: int = 42) -> list:
    """Proportional stratified sample from pool, capped at n."""
    if not pool or n <= 0:
        return []
    n = min(n, len(pool))

    by_cat = defaultdict(list)
    for r in pool:
        by_cat[r[category_key]].append(r)

    # Proportional allocation
    total = len(pool)
    allocations = {cat: max(1, round(n * len(items) / total)) for cat, items in by_cat.items()}

    # Adjust to exactly n
    diff = sum(allocations.values()) - n
    cats_sorted = sorted(allocations, key=lambda c: -len(by_cat[c]))
    for i, cat in enumerate(cats_sorted):
        if diff == 0:
            break
        delta = 1 if diff > 0 else -1
        allocations[cat] = max(0, allocations[cat] - delta)
        diff -= delta if diff > 0 else -delta

    sampled = []
    rng = random.Random(seed)
    for cat, items in by_cat.items():
        k = min(allocations[cat], len(items))
        sampled.extend(rng.sample(items, k))

    return sampled


# ── Sample: fill 100 from tier1 first, supplement from tier2, then tier3 ─────
n_target = N_SAMPLES

sample1 = stratified_sample(tier1, n=min(n_target, len(tier1)))
remaining = n_target - len(sample1)

sample2 = stratified_sample(tier2, n=min(remaining, len(tier2))) if remaining > 0 else []
remaining -= len(sample2)

sample3 = stratified_sample(tier3, n=min(remaining, len(tier3))) if remaining > 0 else []

selected = sample1 + sample2 + sample3
random.shuffle(selected)  # randomise order before writing

print(f"\nSample composition:")
print(f"  From tier 1 (FD + CQ) : {len(sample1)}")
print(f"  From tier 2 (FD or CQ): {len(sample2)}")
print(f"  From tier 3 (OQ only) : {len(sample3)}")
print(f"  Total                 : {len(selected)}")

# ── Check no duplicates ───────────────────────────────────────────────────────
ids = [r["case_id"] for r in selected]
assert len(ids) == len(set(ids)), "Duplicate case_ids in sample!"
print("No duplicate case_ids — check PASSED.")

Tier 1 (FD + CQ)  : 92
Tier 2 (FD or CQ) : 926
Tier 3 (OQ only)  : 977

Sample composition:
  From tier 1 (FD + CQ) : 92
  From tier 2 (FD or CQ): 8
  From tier 3 (OQ only) : 0
  Total                 : 100
No duplicate case_ids — check PASSED.


## 5. Dataset Inspection — Sample Examples

In [6]:
import pandas as pd
from IPython.display import display

df = pd.DataFrame(selected)

print(f"Dataset shape: {df.shape}")
print(f"\nCategory distribution:")
display(df["category"].value_counts().head(20))

print(f"\nAnswer source breakdown:")
display(df["answer_source"].value_counts())

print(f"\nSimulator context richness:")
print(f"  Has FD context  : {df['has_fd'].sum()} ({df['has_fd'].mean():.1%})")
print(f"  Has agent CQ    : {df['has_agent_cq'].sum()} ({df['has_agent_cq'].mean():.1%})")

print(f"\nText length stats:")
df["oq_words"]   = df["original_question"].apply(lambda x: len(x.split()))
df["ctx_words"]  = df["simulator_context"].apply(lambda x: len(x.split()) if x else 0)
df["ans_words"]  = df["accepted_answer"].apply(lambda x: len(x.split()))
display(df[["oq_words", "ctx_words", "ans_words"]].describe().round(1))

Dataset shape: (100, 10)

Category distribution:


category
PowerPoint               17
Excel                    14
Apps_Windows_10           8
Word                      7
Bing_Apps                 6
Bing                      6
Windows_8.1               5
Bing_Search               5
Bing_Maps                 5
Skype_iOS                 5
Windows_7                 4
Windows_10                4
Skype_Android             3
Windows_RT_8.1            3
Skype_Windows_Desktop     2
Skype_Windows_10          2
Outlook_Email             2
Games_Windows_10          1
Skype_Mac                 1
Name: count, dtype: int64


Answer source breakdown:


answer_source
explicit_is_answer    85
fallback_last_pa      15
Name: count, dtype: int64


Simulator context richness:
  Has FD context  : 100 (100.0%)
  Has agent CQ    : 92 (92.0%)

Text length stats:


,oq_words,ctx_words,ans_words
count,100.0,100.0,100.0
mean,83.1,157.7,88.2
std,61.6,114.9,81.8
min,15.0,21.0,6.0
25%,42.5,81.0,48.8
50%,67.5,117.5,66.5
75%,107.2,203.2,114.2
max,341.0,615.0,699.0


In [7]:
# ── Spot-check 3 examples ────────────────────────────────────────────────────
for rec in selected[:3]:
    print("=" * 70)
    print(f"case_id  : {rec['case_id']}")
    print(f"title    : {rec['title']}")
    print(f"category : {rec['category']}")
    print(f"answer_source: {rec['answer_source']}")
    print()
    print("ORIGINAL QUESTION:")
    print(rec["original_question"][:400])
    print()
    if rec["simulator_context"]:
        print("SIMULATOR CONTEXT (first 400 chars):")
        print(rec["simulator_context"][:400])
    else:
        print("SIMULATOR CONTEXT: (empty — OQ-only dialog)")
    print()
    print("ACCEPTED ANSWER (first 300 chars):")
    print(rec["accepted_answer"][:300])
    print()

case_id  : 3529
title    : icon deleted from task bar
category : Windows_8.1
answer_source: explicit_is_answer

ORIGINAL QUESTION:
I accidently deleted an icon for microsoft solitaire from my task bar.   Is there a way to get it back?

SIMULATOR CONTEXT (first 400 chars):
[Further details provided by the user]
windows 8.1

---

[Clarifying questions the support agent previously asked]
And your Windows version is........?

ACCEPTED ANSWER (first 300 chars):
Hi,  This covers how to get a Desktop Shortcut and get your Taskbar icon back:     Left click on far left Icon in the Taskbar > then in the "Start" Window click on down Arrow at bottom left to go to "Apps by Category" Window > locate the App > right click on it and select "Open file location" > in t

case_id  : 22873
title    : How can I restore the old version of the calculator on Windows 8.1?
category : Windows_8.1
answer_source: explicit_is_answer

ORIGINAL QUESTION:
Original title: The old version of calculator I have recently up

## 6. Simulator Context Completeness Check

For dialogs where `simulator_context` is empty (OQ-only tier-3 cases), the simulator has no hidden context to draw from when answering a clarifying question. We flag these — they are still usable (the simulator must improvise based on OQ alone) but they're lower quality.

We also compute a richness score for transparency.

In [8]:
empty_ctx = [r for r in selected if not r["simulator_context"].strip()]
print(f"Dialogs with empty simulator context: {len(empty_ctx)} / {len(selected)}")

if empty_ctx:
    print("\nThese cases have OQ + accepted answer but no FD or agent CQ.")
    print("The simulator will respond based on OQ context alone.")
    for r in empty_ctx[:3]:
        print(f"  [{r['case_id']}] {r['title'][:60]}")

# ── Richness score (0–2) for each record ─────────────────────────────────────
for r in selected:
    r["context_richness"] = int(r["has_fd"]) + int(r["has_agent_cq"])

from collections import Counter
richness_dist = Counter(r["context_richness"] for r in selected)
print(f"\nContext richness distribution (0=none, 1=FD or CQ, 2=both):")
for score in sorted(richness_dist):
    print(f"  Score {score}: {richness_dist[score]} records")

Dialogs with empty simulator context: 0 / 100

Context richness distribution (0=none, 1=FD or CQ, 2=both):
  Score 1: 8 records
  Score 2: 92 records


## 7. LLM Synthesis — User Situation Summary

**Why raw FD utterances are not enough**

The user's FD utterances were written in response to the *original agent's specific questions* in that dialog. When our model generates a *different* clarifying question, the simulator must answer using context that wasn't written for it. Raw FD is narrow and conversational — full of greetings, redundant phrasing, and implicit references that don't generalise.

**What we do instead**

For each record, we send the full available information (OQ + raw FD + agent CQ context) to Gemini and ask it to synthesise a single, structured **User Situation Summary** — a clean background document the simulator can query freely to answer any CQ.

The output format is deliberately structured (labelled sections) so the simulator can locate specific facts instantly when answering questions like *"What OS are you on?"* or *"What error message do you see?"*

Synthesis runs at 1 s/call with resume support — safe to interrupt and restart.

In [ ]:
import time
import importlib
from pathlib import Path

sys.path.insert(0, str(ROOT))

import src.providers as providers_mod
import src.utils as utils_mod
importlib.reload(providers_mod)
importlib.reload(utils_mod)

from src.providers import GeminiProvider
from src.utils import load_dotenv
from config import SIMULATOR_MODEL_ID

load_dotenv(ROOT / ".env")

SYNTHESIS_CACHE = OUT_DIR / "msdialog_synthesis_cache.jsonl"
SYNTHESIS_INTERVAL = 1.2   # seconds between API calls

SYNTHESIS_SYSTEM = """You are a technical documentation assistant. Your job is to read a messy tech-support conversation and rewrite the user's situation as a clean, structured background document.

The document will be used by a virtual user (simulator) to answer any follow-up questions about their problem. It must:
- Be written in first person ("I", "my") — the simulator speaks AS the user
- Capture every concrete technical fact from the conversation
- Be organised under clear section headers so specific facts are easy to find
- Strip all greetings, thanks, and conversational filler
- Never include the solution or anything the agent suggested

Output ONLY the structured summary, no preamble or explanation."""

SYNTHESIS_USER_TEMPLATE = """Here is a tech-support thread. Synthesise a structured User Situation Summary.

PRODUCT / CATEGORY: {category}
TITLE: {title}

USER'S ORIGINAL QUESTION:
{original_question}

ADDITIONAL INFORMATION FROM THE CONVERSATION:
{raw_context}

---
Write the User Situation Summary using these sections (omit a section if there is no information for it):

CORE PROBLEM
[One or two sentences describing exactly what is broken or not working]

SYSTEM & SOFTWARE INFO
[OS version, device type, software name and version, browser, app version — any specifics mentioned]

SYMPTOMS & ERROR DETAILS
[Exact error messages, what the user sees on screen, how often it happens, when it started]

WHAT I HAVE ALREADY TRIED
[Any steps the user already attempted before asking for help]

ADDITIONAL CONTEXT
[Any other relevant facts: network setup, other devices affected, recent changes, workarounds discovered]"""


def synthesise_one(provider: GeminiProvider, record: dict) -> str:
    """Call the LLM to synthesise a structured user situation summary."""
    raw_ctx = record["simulator_context"].strip()
    if not raw_ctx:
        raw_ctx = "(No additional details were provided beyond the original question.)"

    user_msg = SYNTHESIS_USER_TEMPLATE.format(
        category=record["category"],
        title=record["title"],
        original_question=record["original_question"],
        raw_context=raw_ctx,
    )
    return provider.call(
        system_instruction=SYNTHESIS_SYSTEM,
        user_message=user_msg,
        temperature=0.0,
        max_tokens=5000,
    )


# ── Load existing cache (resume support) ─────────────────────────────────────
cache: dict[str, str] = {}
if SYNTHESIS_CACHE.exists():
    with open(SYNTHESIS_CACHE, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                entry = json.loads(line)
                cache[entry["case_id"]] = entry["synthesised_context"]
    print(f"Loaded {len(cache)} cached syntheses from {SYNTHESIS_CACHE.name}")
else:
    print("No cache found — starting fresh.")

# ── Run synthesis ─────────────────────────────────────────────────────────────
provider = GeminiProvider(model_id=SIMULATOR_MODEL_ID)

# Smoke test — 5000 tokens covers gemini-3.1-pro-preview's internal thinking
# budget before it generates the actual output
_smoke = provider.call(
    system_instruction="You are a helpful assistant.",
    user_message="Reply with exactly: SYNTHESIS READY",
    temperature=0.0, max_tokens=5000,
)
assert _smoke and "SYNTHESIS" in _smoke.upper(), f"Smoke test failed: {_smoke!r}"
print(f"Provider smoke test PASSED: {_smoke.strip()}")

to_process = [r for r in selected if r["case_id"] not in cache]
print(f"\nRecords to synthesise: {len(to_process)} / {len(selected)} (skipping {len(cache)} cached)")

cache_file = open(SYNTHESIS_CACHE, "a", encoding="utf-8")

try:
    for i, rec in enumerate(to_process, 1):
        synthesised = synthesise_one(provider, rec)
        cache[rec["case_id"]] = synthesised
        cache_file.write(json.dumps({"case_id": rec["case_id"], "synthesised_context": synthesised}, ensure_ascii=False) + "\n")
        cache_file.flush()
        print(f"[{i:3d}/{len(to_process)}] {rec['case_id']} ({rec['category']}) — {len(synthesised.split())} words")
        if i < len(to_process):
            time.sleep(SYNTHESIS_INTERVAL)
finally:
    cache_file.close()

print(f"\nSynthesis complete. Total cached: {len(cache)}")


In [ ]:
# ── Attach synthesised context to records + spot-check ───────────────────────
for rec in selected:
    rec["synthesised_context"] = cache[rec["case_id"]]

# Spot-check 2 examples: raw vs synthesised
for rec in selected[:2]:
    print("=" * 70)
    print(f"[{rec['case_id']}] {rec['title']} ({rec['category']})")
    print()
    print("── RAW context (what we had before) ──")
    print(rec["simulator_context"][:400])
    print()
    print("── SYNTHESISED context ──")
    print(rec["synthesised_context"][:600])
    print()

# ── Contamination check on synthesised context ───────────────────────────────
contaminated = []
for rec in selected:
    ans_words = rec["accepted_answer"].lower().split()
    ctx_lower = rec["synthesised_context"].lower()
    for i in range(len(ans_words) - 9):
        chunk = " ".join(ans_words[i:i+10])
        if chunk in ctx_lower:
            contaminated.append(rec["case_id"])
            break

print(f"Contamination check on synthesised context (10-word chunks): "
      f"{len(contaminated)}/100 — {'CLEAN' if not contaminated else contaminated}")


## 7. Save Output

Write `msdialog_100.jsonl` — one JSON object per line, ordered fields matching MedQA schema where applicable.

In [ ]:
# ── Assign sequential case_ids for cleaner downstream referencing ─────────────
output_records = []
for i, rec in enumerate(selected, start=1):
    output_records.append({
        # ── Core experiment fields (what the pipeline reads) ──────────────────
        "case_id":            f"msd_{i:03d}",
        "title":              rec["title"],
        "category":           rec["category"],
        "original_question":  rec["original_question"],
        # synthesised_context replaces raw simulator_context —
        # this is what the PatientSimulator receives
        "simulator_context":  rec["synthesised_context"],
        "accepted_answer":    rec["accepted_answer"],

        # ── Provenance & quality metadata ─────────────────────────────────────
        "source_dialog_id":   rec["case_id"],          # original MSDialog ID
        "answer_source":      rec["answer_source"],    # explicit_is_answer | fallback_last_pa
        "context_richness":   rec["context_richness"], # 0–2 raw richness before synthesis
        "n_utterances":       rec["n_utterances"],
        "has_fd":             rec["has_fd"],
        "has_agent_cq":       rec["has_agent_cq"],
        # keep raw context for inspection / ablation
        "raw_simulator_context": rec["simulator_context"],
    })

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for rec in output_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Saved {len(output_records)} records to {OUTPUT_JSONL}")

# ── Verify round-trip ─────────────────────────────────────────────────────────
with open(OUTPUT_JSONL, encoding="utf-8") as f:
    loaded = [json.loads(l) for l in f if l.strip()]

assert len(loaded) == N_SAMPLES, f"Expected {N_SAMPLES}, got {len(loaded)}"
assert all("simulator_context" in r for r in loaded)
assert all(r["simulator_context"].strip() for r in loaded), "Some synthesised contexts are empty!"
print(f"Round-trip verification PASSED — {len(loaded)} records, all with non-empty simulator_context.")
print(f"\nFields: {list(loaded[0].keys())}")


## 8. Final Summary

In [10]:
out_df = pd.DataFrame(output_records)

print("=" * 60)
print("MS-DIALOG DATASET SUMMARY")
print("=" * 60)
print(f"Total records        : {len(out_df)}")
print(f"Unique categories    : {out_df['category'].nunique()}")
print()
print("Simulator context richness:")
for score, count in sorted(Counter(out_df['context_richness']).items()):
    labels = {0: 'none (OQ only)', 1: 'FD or agent CQ', 2: 'FD + agent CQ'}
    print(f"  {labels[score]:20s}: {count:3d} ({count/len(out_df):.1%})")
print()
print("Answer sources:")
for src, count in Counter(out_df['answer_source']).items():
    print(f"  {src:25s}: {count:3d} ({count/len(out_df):.1%})")
print()
print("Word counts (mean):")
print(f"  original_question   : {out_df['original_question'].apply(lambda x: len(x.split())).mean():.0f} words")
print(f"  simulator_context   : {out_df['simulator_context'].apply(lambda x: len(x.split()) if x else 0).mean():.0f} words")
print(f"  accepted_answer     : {out_df['accepted_answer'].apply(lambda x: len(x.split())).mean():.0f} words")
print()
print(f"Output file          : {OUTPUT_JSONL}")
print("=" * 60)

MS-DIALOG DATASET SUMMARY
Total records        : 100
Unique categories    : 19

Simulator context richness:
  FD or agent CQ      :   8 (8.0%)
  FD + agent CQ       :  92 (92.0%)

Answer sources:
  explicit_is_answer       :  85 (85.0%)
  fallback_last_pa         :  15 (15.0%)

Word counts (mean):
  original_question   : 83 words
  simulator_context   : 158 words
  accepted_answer     : 88 words

Output file          : D:\final_project\pilot_study\datasets\ms-dialog\msdialog_100.jsonl


## Experiment Design Notes

### How this maps to the Phase 1 pipeline

```
Model sees:
  [title]  — product context (e.g. "Windows 10 — Edge crashes")
  [original_question]  — user's problem description (OQ utterances)
  Prompt: "What is the most helpful solution to this user's problem?"

Model generates:
  preliminary_solution  — initial attempt at resolving the issue
  clarifying_question   — optional CQ to the user
  confidence            — self-reported certainty

Simulator receives:
  simulator_context     — FD utterances + agent CQs (hidden from model)
  clarifying_question   — what the model asked

Evaluator compares:
  final_solution (after CQ rounds) vs accepted_answer
  → binary RESOLVED / NOT RESOLVED verdict
```

### Differences from MedQA
| Aspect | MedQA | MS-Dialog |
|---|---|---|
| Task type | Diagnosis MCQ | Open-ended tech support |
| Ground truth | Correct option (A/B/C/D) | Accepted answer text |
| Evaluation | Exact match | Semantic RESOLVED/NOT RESOLVED |
| Difficulty | Labelled (easy/medium/hard) | Implicit (by category + dialog length) |
| Domain | Clinical medicine | Microsoft tech support |